- Notebook will read data from the races.csv file and will ingest into bronze schema(raw data)
- After applying transformations, will load the dataframe to a delta table in silver schema( ready to use data)
- Based on business requirement will preform aggregation and save the data to gold schema


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
races_schema = StructType(fields=[StructField("raceId", IntegerType(), False),
                                   StructField("year", IntegerType(), True),
                                   StructField("round", IntegerType(), True),
                                   StructField("circuitId", IntegerType(), True),
                                   StructField("name", StringType(), True),
                                   StructField("date", StringType(), True),
                                   StructField("time", StringType(), True),
                                   StructField("url", StringType(), True)
                                  ])
races_df = spark.read\
    .option("header", True)\
    .schema(races_schema)\
    .csv('/Volumes/workspace/bronze/raw_files/races.csv')
races_df.printSchema()
races_df.show()

In [0]:
%sql
select current_schema();
use  schema bronze;

In [0]:
races_df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("races_raw")

In [0]:
races_transform_silver_df  = spark.read.table("races_raw")
display(races_transform_silver_df)


In [0]:
#concat date and time columns
from pyspark.sql.functions import concat_ws, lit,to_timestamp,col
races_transform_silver_df =races_transform_silver_df.withColumn(
    "race_timestamp", 
    (concat_ws(" ","date","time"))
)

races_transform_silver_df.show()

In [0]:
races_transform_silver_df = races_transform_silver_df.drop("date","time","url")
display(races_transform_silver_df)

In [0]:
%sql
SELECT CURRENT_SCHEMA();
USE SCHEMA silver;

In [0]:
races_transform_silver_df.write\
    .format("delta")\
    .partitionBy("year")\
    .mode("overwrite")\
    .saveAsTable("races_info")

In [0]:
races_gld_df = spark.read.table("races_info")
display(races_gld_df)

In [0]:
from pyspark.sql import functions as F
races_gld_df_final = races_gld_df.groupBy("year").agg(
    F.count("raceId").alias("races_count")
).orderBy("year")
display(races_gld_df_final)

In [0]:
%sql
USE SCHEMA silver;
select current_schema();

In [0]:
races_gld_df_final.write.saveAsTable("gold.races_gld")

In [0]:
%sql
DESCRIBE HISTORY gold.races_gld